# NLP con Long-Short Term Memory (LSTM)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ohtar10/icesi-nlp/blob/main/Sesion2/2-nlp-with-lstm.ipynb)

En este notebook implementaremos un clasificador de noticias en español utilizando la arquitectura de red LSTM. La idea es tener un punto de referencia para comparar cuando observemos la parte de transformers, por lo que utilizaremos el mismo dataset y tarea de ejemplo. Utilizarémos las utilidades de tokenización de huggingface transformers para ayudarnos con esta tarea.

#### Referencias
- Dataset: https://huggingface.co/datasets/MarcOrfilaCarreras/spanish-news
- [Long Short-Term Memory](https://www.researchgate.net/publication/13853244_Long_Short-Term_Memory#fullTextFileContent)

In [1]:
import importlib.metadata
installed_packages = [
    dist.metadata['Name'].lower() 
    for dist in importlib.metadata.distributions() 
    if dist.metadata['Name'] is not None
]
IN_COLAB = 'google-colab' in installed_packages

In [2]:
#!test '{IN_COLAB}' = 'True' && wget  https://github.com/Ohtar10/icesi-nlp/raw/refs/heads/main/requirements.txt && pip install -r requirements.txt
!test '{IN_COLAB}' = 'True' && sudo apt-get update -y
!test '{IN_COLAB}' = 'True' && sudo apt-get install python3.10 python3.10-distutils python3.10-lib2to3 -y
!test '{IN_COLAB}' = 'True' && sudo update-alternatives --install /usr/local/bin/python python /usr/bin/python3.11 2
!test '{IN_COLAB}' = 'True' && sudo update-alternatives --install /usr/local/bin/python python /usr/bin/python3.10 1
!test '{IN_COLAB}' = 'True' && pip install lightning datasets

"test" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.
"test" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.
"test" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.
"test" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.
"test" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


### Cargando el dataset
Este es un dataset pequeño de articulos de noticias en idioma español con sus respectivas categorías. El dataset está disponible en el HuggingFace Hub y puede ser fácilmente descargado con la librería.

In [3]:
from datasets import load_dataset
import warnings
import os

warnings.filterwarnings("ignore")
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
dataset = load_dataset('MarcOrfilaCarreras/spanish-news', split='train')
dataset

Dataset({
    features: ['language', 'category', 'newspaper', 'hash', 'text'],
    num_rows: 10200
})

Observemos uno de sus registros

In [4]:
dataset[1]

{'language': 'es',
 'category': 'play',
 'newspaper': 'de_lector_a_lector',
 'hash': 'b387bc0a5ad68524c8aa5da489555ca41d5a3575',
 'text': 'El coraje de ser, de Mónica Cavallé, la aventura del autoconocimiento filosófico.Todos experimentamos momentos de plenitud vinculados a la expresión directa y auténtica de nosotros mismos: momentos de contemplación de la belleza del mundo en que nuestros sentidos se abren como si lo vieran por primera vez, de intimidad y comunión con otro ser humano, de fluidez creativa, de expresión confiada y libre… Estos momentos permiten intuir lo que puede ser una vida en la que no meramente se existe, sino en la que se vive en todo el sentido de esta palabra.Esta vida solo es posible cuando sabemos quiénes somos, cuando nos conocemos a nosotros mismos de modo experiencial: no cuando nos llenamos de ideas sobre nosotros, sino cuando nos asentamos en nuestro ser real, más allá de nuestras defensas, máscaras y falsos yoes.El coraje de ser es una invitación a aden

Para los efectos de esta tarea, nos servirán el texto y la categoría naturalmente.

A manera general, observemos que tan largos o cortos tienden a ser los textos.

In [5]:
text_lengths = [len(row['text']) for row in dataset]
print(f"Texto más corto: {min(text_lengths)}")
print(f"Texto más largo: {max(text_lengths)}")
print(f"Longitud promedio: {sum(text_lengths) / len(text_lengths)}")

Texto más corto: 501
Texto más largo: 204324
Longitud promedio: 4218.154509803921


Estos valores son la cantidad de *caractéres* que tiene las secuencias. Una decisión ingenua pero útil en este momento podría ser ajustar la longitud de las secuencias que vamos a usar para el entrenamiento a unos 2000 tokens. Esto podría ser suficiente para capturar una porción significativa de los textos.

## Definiendo el Tokenizer

Ahora, vamos a definir el tokenizer para nuestra tarea. Para mantener las cosas simples, vamos a mantener un conteo de palabras y vamos a hacer un corte hasta los primeros 50mil tokens.

In [6]:
import re
from collections import Counter

def simple_tokenizer(text):
    text = text.lower()
    text = re.sub(r"[^a-záéíóúüñ]+", " ", text)
    return text.strip().split()

# Construimos el vocabulario a partir de conjunto de datos.
token_counts = Counter()
for text in dataset["text"]:
    token_counts.update(simple_tokenizer(text))

# 50k-2 porque necesitamos reservar espacio para los dos tokens especiales
top_n_tokens = list(token_counts.keys())[:50000-2]
vocab = {"[PAD]": 0, "[UNK]": 1}
for token in top_n_tokens:
    vocab[token] = len(vocab)

def tokenize_text(text, max_length=50):
    tokens = simple_tokenizer(text)
    ids = [vocab.get(tok, vocab["[UNK]"]) for tok in tokens[:max_length]]
    ids += [vocab["[PAD]"]] * (max_length - len(ids))
    return ids

Exploremos ahora el tokenizador obtenido.

In [7]:
print(f"Vocabulario: {len(vocab)} tokens")
print("Primeros 15 tokens:")
print(f"{top_n_tokens[:15]}")
print("15 tokens de en medio:")
print(f"{top_n_tokens[1000:1015]}")
print("Últimos 15 tokens:")
print(f"{top_n_tokens[-15:]}")

Vocabulario: 50000 tokens
Primeros 15 tokens:
['valladolid', 'misteriosa', 'es', 'el', 'título', 'del', 'nuevo', 'libro', 'que', 'acaba', 'de', 'publicar', 'la', 'editorial', 'almuzara']
15 tokens de en medio:
['trabajar', 'primero', 'seis', 'meses', 'guionista', 'antigua', 'city', 'tv', 'después', 'llamó', 'tres', 'medio', 'redactora', 'super', 'pop']
Últimos 15 tokens:
['risco', 'infecciosas', 'amalia', 'simoni', 'camagüey', 'instructor', 'proteinuria', 'disfunción', 'endotelial', 'diabéticos', 'amlodipine', 'bloqueador', 'aml', 'hctz', 'tolerada']


Esta forma de exploración es para darnos una idea de las palabras más utilizadas en el corpus y nos dará un indicio de si la tokenización es adecuada o no. Vemos que tenemos algunos stop words, como artículos (el, la) y conectores (del, que). Para una tarea de clasificación de texto podríamos prescindir de estos pero para facilitar las cosas y ya que los demás tokens lucen bien, podemos preservarlos.

Ahora veamos como convierte el tokenizador una oración muy sencilla:

In [8]:
tokenized = tokenize_text("hola mundo", max_length=8)
tokenized

[29340, 157, 0, 0, 0, 0, 0, 0]

Lo que obtenemos de vuelta son los ids de cada token según el vocabulario. Ahora algo importante que notamos aquí es el *padding*, durante el entrenamiento, queremos que las secuencias sean de tamaño fijo, para asi operar comodamente con matrices. Pero ya vimos que no todos los textos tienen la misma longitud. Entonces que hacer? para los que son más largos que una longitud dada simplemente cortamos, pero para los que son más cortos, debemos *rellenar* lo faltante con un *token especial de relleno o padding*. Y es justo lo que definimos allí, cuando la cadena es inferior a 8 **tokens**, entonces debemos hacer padding hasta que se cumplan los 8.

Si queremos saber a que token exactamente hacen referencia estos ids, simplemente revisamos el vocabulario que hemos construido:

In [9]:
id_2_token = {v: k for k, v in vocab.items()}
[id_2_token[token] for token in tokenized]

['hola', 'mundo', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']

Claramente vemos los 3 tokens como cadenas independientes (el padding se considera un token independiente).

### Definiendo el dataset de pytorch
Ahora podemos proceder a definir el dataset. Esto debería ser muy sencillo dado que nuestro dataset es pequeño y ya tenemos el tokenizador listo.

In [10]:
import torch
import numpy as np
from typing import Tuple, Dict
from torch.utils.data import Dataset

class SpanishNewsDataset(Dataset):

    def __init__(self, tokenizer, dataset, seq_length: int = 512):
        self.tokenizer = tokenizer
        self.dataset = dataset
        self.seq_length = seq_length
        # Definimos estos dos mapas para facilitarnos la tarea
        # de traducir de nombres de categoría a ids de categoría.
        self.id_2_class_map = dict(enumerate(np.unique(dataset[:]['category'])))
        self.class_2_id_map = {v: k for k, v in self.id_2_class_map.items()}
        self.num_classes = len(self.id_2_class_map)

    def __getitem__(self, index) -> Dict[str, torch.Tensor]:
        text, y = self.dataset[index]['text'], self.dataset[index]['category']
        y = self.class_2_id_map[y]
        data = {'input_ids': torch.tensor(self.tokenizer(text, max_length=self.seq_length))}
        data['y'] = torch.tensor(y)
        return data


    def __len__(self):
        return len(self.dataset)

Ahora instanciaremos el dataset entero. Para este experimento, definiremos un tamaño máximo de secuencia de 2048 **tokens**. Que según nuestra intuición arriba, debería ser suficiente para la tarea.

In [11]:
max_len = 512 
spanish_news_dataset = SpanishNewsDataset(tokenize_text, dataset, seq_length=max_len)
assert len(spanish_news_dataset) == len(dataset)

Y luego, procedemos a hacer el train-val-test split y crear los dataloaders.

In [12]:
from torch.utils.data import random_split
from torch.utils.data import DataLoader

batch_size = 4 if not IN_COLAB else 12
train_dataset, val_dataset, test_dataset = random_split(spanish_news_dataset, lengths=[0.8, 0.1, 0.1])
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=0)

## Definición del modelo LSTM

Ahora vamos a configurar un módulo pytorch simple para este problema. Vamos ha utilizar los embeddings, que vendrían siendo los vectores de palabra. Pytorch nos ofrece una capa con la que directamente podemos entrenarlos a partir de los token ids obtenidos. El resto consistirá en invocar una capa LSTM seguida de una capa densa para la clasificación.

Recordemos que las redes recurrentes como las LSTM por diseño enlazan todas las dimensiones del vector de entrada, formando así la secuencia, la estructura natural que necesitamos representar.

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LSTMBlock(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, num_layers=2, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout)
    
    def forward(self, x):
        embedded = self.embedding(x)
        output, (hidden, _) = self.lstm(embedded)
        return hidden[-1]


### Definición del clasificador

Finalmente, definimos el modelo en si. Este modelo constará de 3 capas:

- La tokenización, tal como la definimos anteriormente.
- El bloque LSTM, que acabamos de decinir.
- Una capa densa adicional que servirá como clasificador de aquello que nos entregue la capa del transformer.

Como este es un LightningModule, aquí definiremos el resto de funciones utilitarias para el entrenamiento de la tarea.

In [14]:
from pytorch_lightning import LightningModule, Trainer
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from torchmetrics import Accuracy

class SpanishNewsClassifierWithLSTM(LightningModule):

    def __init__(self, vocab_size: int, num_classes: int, emb_dim: int, hidden_dim: int = 128):
        super(SpanishNewsClassifierWithLSTM, self).__init__()
        self.num_classes = num_classes
        self.lstm = LSTMBlock(vocab_size, emb_dim, hidden_dim, num_classes)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(hidden_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes),
            nn.LogSoftmax(dim=1)
        )

        self.train_acc = Accuracy(task='multiclass', num_classes=num_classes)
        self.val_acc = Accuracy(task='multiclass', num_classes=num_classes)
        self.test_acc = Accuracy(task='multiclass', num_classes=num_classes)

    def forward(self, x):
        embeddings = self.lstm(x)
        return self.classifier(embeddings)

    
    def training_step(self, batch, batch_idx):
        x, y = batch['input_ids'], batch['y']
        # print(f"\nbatch-idx: {batch_idx}")
        # print(f"shape of x: {x.shape}")
        # print(torch.max(x, dim=0))
        y_hat = self(x)
        loss = F.cross_entropy(y_hat, y)
        self.train_acc(y_hat, y)
        self.log('train-loss', loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log('train-acc', self.train_acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss
    
    def validation_step(self, batch):
        x, y = batch['input_ids'], batch['y']
        y_hat = self(x)
        loss = F.cross_entropy(y_hat, y)
        self.val_acc(y_hat, y)
        self.log('val-loss', loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log('val-acc', self.val_acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss
    
    def test_step(self, batch):
        x, y = batch['input_ids'], batch['y']
        y_hat = self(x)
        self.test_acc(y_hat, y)
        self.log('test-acc', self.test_acc, prog_bar=True, on_step=False, on_epoch=True)


    def predict_step(self, batch):
        x = batch['input_ids']
        return self(x)


    def configure_optimizers(self):
        optimizer =  torch.optim.AdamW(self.parameters(), lr=1e-3, weight_decay=1e-5)
        return optimizer

    
model = SpanishNewsClassifierWithLSTM(vocab_size=len(vocab) + 1, num_classes=spanish_news_dataset.num_classes, emb_dim=256)

tb_logger = TensorBoardLogger('tb_logs', name='LSTMClassifier')
callbacks=[EarlyStopping(monitor='train-loss', patience=3, mode='min')]
trainer = Trainer(max_epochs=10, devices=1, logger=tb_logger, callbacks=callbacks, precision=32)

trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ lstm       │ LSTMBlock          │ 13.1 M │ train │     0 │
│ 1 │ classifier │ Sequential         │  200 K │ train │     0 │
│ 2 │ train_acc  │ MulticlassAccuracy │      0 │ train │     0 │
│ 3 │ val_acc    │ MulticlassAccuracy │      0 │ train │     0 │
│ 4 │ test_acc   │ MulticlassAccuracy │      0 │ train │     0 │
└───┴────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 13.3 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 13.3 M                                                                                               
Total estimated model params size (MB): 53                                                                         
Modules in train mode: 16                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=10` reached.


Observemos el proceso de entrenamiento

In [15]:
%load_ext tensorboard

In [16]:
%tensorboard --logdir tb_logs/

Reusing TensorBoard on port 6006 (pid 33572), started 1:07:42 ago. (Use '!kill 33572' to kill it.)

Y como es de esperarse, realizaremos la validación contra el conjunto de prueba.

In [17]:
model.eval()
trainer.test(model, test_loader)

Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test-acc          │    0.9205882549285889     │
└───────────────────────────┴───────────────────────────┘

[{'test-acc': 0.9205882549285889}]

### Haciendo predicciones

Finalmente, vamos a hacer uso del modelo y ver que tan bueno es para la clasificación de noticias.

In [18]:
predictions = trainer.predict(model, test_loader)
predictions = torch.cat(predictions, dim=0)
predictions = torch.argmax(predictions, dim=-1)
predictions = [spanish_news_dataset.id_2_class_map[pred] for pred in predictions.numpy()]

Output()

In [19]:
import pandas as pd

test_indices = test_dataset.indices
df = pd.DataFrame(data={
    "texto": dataset[test_indices]['text'],
    "tokens": [tokenize_text(v) for v in dataset[test_indices]['text']],
    "categoría": dataset[test_indices]['category'],
    'predicción': predictions
}, index=test_indices)

df['tokens_string'] = df.tokens.apply(lambda t: ' '.join([id_2_token[i] for i in t]))
df = df[["texto", "tokens", "tokens_string", "categoría", "predicción"]]
df.style.set_table_styles(
    [
        {'selector': 'td', 'props': [('word-wrap', 'break-word')]}
    ]
)
df.head(15)

,texto,tokens,tokens_string,categoría,predicción
1850,Resumen:Se realizó un estudio descriptivo tran...,"[8878, 17, 5339, 102, 1608, 49568, 24511, 54, ...",resumen se realizó un estudio descriptivo tran...,medicine,medicine
6091,Mauricio Macri quiere aprovechar la crisis de ...,"[26663, 1, 1762, 2587, 14, 1580, 12, 5944, 1, ...",mauricio [UNK] quiere aprovechar la crisis de ...,politics,politics
593,Este 2023 hemos podido ver varias películas de...,"[55, 4227, 6108, 1938, 1591, 2338, 12, 3870, 1...",este hemos podido ver varias películas de supe...,play,play
3628,Memphis Grizzlies ha vivido en la cresta de la...,"[1, 1, 260, 1327, 54, 14, 1, 12, 14, 7747, 191...",[UNK] [UNK] ha vivido en la [UNK] de la ola es...,sport,sport
3398,La Agencia de Ciencia y Tecnología de la Regió...,"[14, 11130, 12, 4182, 36, 3328, 12, 14, 16731,...",la agencia de ciencia y tecnología de la regió...,astronomy,military
1848,"Autores por orden:1.- Margarita García Huerta,...","[2097, 45, 2897, 48141, 13329, 46483, 35849, 2...",autores por orden margarita garcía huerta adju...,medicine,religion
3100,"En el vasto rincón del sistema solar, Tritón, ...","[54, 5, 1, 8042, 7, 1170, 32576, 24861, 14, 20...",en el [UNK] rincón del sistema solar tritón la...,astronomy,astronomy
7052,La Agencia de Investigación de Proyectos Avanz...,"[14, 11130, 12, 156, 12, 3711, 15893, 12, 1327...",la agencia de investigación de proyectos avanz...,military,military
2053,Comportamiento de la crisis hipertensiva. Cent...,"[11893, 12, 14, 1580, 1, 3362, 11900, 12, 1167...",comportamiento de la crisis [UNK] centro médic...,medicine,medicine
3897,El Oviedo entró con una marcha más en el parti...,"[5, 28179, 1032, 58, 19, 10114, 130, 54, 5, 46...",el oviedo entró con una marcha más en el parti...,sport,sport


In [20]:
errors = df[df['categoría'] != df['predicción']]
errors.head(15)

,texto,tokens,tokens_string,categoría,predicción
3398,La Agencia de Ciencia y Tecnología de la Regió...,"[14, 11130, 12, 4182, 36, 3328, 12, 14, 16731,...",la agencia de ciencia y tecnología de la regió...,astronomy,military
1848,"Autores por orden:1.- Margarita García Huerta,...","[2097, 45, 2897, 48141, 13329, 46483, 35849, 2...",autores por orden margarita garcía huerta adju...,medicine,religion
2949,"La Búsqueda de Inteligencia Extraterrestre, o ...","[14, 915, 12, 2003, 12994, 96, 1, 45, 128, 211...",la búsqueda de inteligencia extraterrestre o [...,astronomy,religion
9588,Mientras que hay quienes tienen la suerte de t...,"[1201, 10, 124, 1251, 440, 14, 3673, 12, 3001,...",mientras que hay quienes tienen la suerte de t...,economy,motor
9138,Se trata del dispositivo EDR (Event Data Recor...,"[17, 18, 7, 7733, 1, 1, 13167, 1, 10, 4, 102, ...",se trata del dispositivo [UNK] [UNK] data [UNK...,motor,economy
4975,El pelo de Sonsoles Ónega es una de sus señas ...,"[5, 11785, 12, 27033, 27034, 4, 19, 12, 128, 5...",el pelo de sonsoles ónega es una de sus señas ...,fashion,alimentation
6124,"Finalmente, y luego de un prolongado debate y ...","[5876, 36, 995, 12, 102, 34526, 7760, 36, 1432...",finalmente y luego de un prolongado debate y f...,politics,religion
8547,"Hoy, en monstruosidades sobre ruedas, tenemos ...","[789, 54, 1, 126, 16813, 2658, 42, 1, 10, 491,...",hoy en [UNK] sobre ruedas tenemos esta [UNK] q...,motor,economy
6787,"Decenas de falangistas, nostálgicos del franqu...","[11820, 12, 1, 27288, 7, 13872, 36, 12446, 12,...",decenas de [UNK] nostálgicos del franquismo y ...,politics,religion
9566,La cadena de grandes almacenes Walmart comprar...,"[14, 11688, 12, 901, 34244, 1, 35652, 40, 14, ...",la cadena de grandes almacenes [UNK] comprará ...,economy,military


## Conclusiones

- En este caso tenemos una tarea de clasificación de texto de múltiples clases.
- Estamos usando un bloque LSTM como featurizer, es decir lo usamos para extraer features de las secuencias de entrada con las cuales harémos predicciones luego.
- Nótese que de las capas LSTM, solo nos interesa la última, ya que esta recupera todas las operaciones enalazadas anteriores.
- Observamos que el modelo toma su tiempo en entrenar, esto es natural debido al diseño de las LSTM, donde por cada paso de tiempo se debe computar un gradiente, por lo que el computo es mucho mayor.